# 01 — Adat-előkészítés (v0 / vKNN / v1)

Ez a notebook generálja a három split-verziót:

| Verzió | Sor | Cél |
|---|---|---|
| **v0** | 200 000 | gyors fejlesztés, sanity check |
| **vKNN** | 500 000 | KNN modell (geostat fit ezen fut) |
| **v1** | ~9,6M | final tanításhoz, KNN nélkül |

Két lépés verziónként: `prepare()` (CSV → train/test parquet) és `build_features()` (parquet → feature-engineered npy + scaler).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Slityak/gepitan-seoul-bike-trip/blob/main/notebooks/01_data_prep.ipynb)

## 1. Setup (Colab/lokális, Drive mount, repo klónozás)

In [ ]:
import os
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    if not os.path.exists('/content/gepitan-beadando'):
        !git clone https://github.com/Slityak/gepitan-seoul-bike-trip.git /content/gepitan-beadando

    os.chdir('/content/gepitan-beadando')
    !git pull
    !pip install -q -r requirements.txt
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')

sys.path.insert(0, '.')
print(f'Working dir: {os.getcwd()}')
print(f'In Colab: {IN_COLAB}')

## 2. Importok és path-ok

A `DATA_DIR` Colabon Drive-ra (`MyDrive/GepiTan_Beadando/data`), lokálisan a repo `data/` mappájára mutat. A nyers `For_modeling.csv` is itt keresendő (vagy a projekt gyökerében).

In [ ]:
from pathlib import Path

from src.config import DATA_DIR, PROJECT_ROOT, SPLITS_DIR, ensure_dirs
from src.prepare_data import prepare, SAMPLE_SIZES
from src.geostat_elemzes import build_features

ensure_dirs()
DATA_DIR.mkdir(parents=True, exist_ok=True)
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'DATA_DIR:     {DATA_DIR}')
print(f'SPLITS_DIR:   {SPLITS_DIR}')
print(f'Verziók:      {list(SAMPLE_SIZES)}')

## 3. Nyers CSV beszerzése (kagglehub)

A `kagglehub` a `~/.cache/kagglehub/`-ba tölti le a datasetet. Ez Colabon nem perzisztens (minden új session újra letölti), **de cserébe a Drive-od tiszta marad** — csak a feldolgozott eredmények kerülnek `DATA_DIR`-be.

**Lokálisan:** kell `~/.kaggle/kaggle.json` (megvan).

**Colabon:** bal oldali 🔑 (Secrets) ikon → vegyél fel két secret-et, és engedélyezd ehhez a notebookhoz:
- `KAGGLE_USERNAME`
- `KAGGLE_KEY`

In [ ]:
if IN_COLAB:
    from google.colab import userdata
    try:
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    except Exception as e:
        raise RuntimeError(
            "Hiányzó Colab secret. Bal oldali 🔑 ikon → vegyél fel "
            "KAGGLE_USERNAME és KAGGLE_KEY secret-et, és engedélyezd ehhez a notebookhoz."
        ) from e

import kagglehub

dataset_dir = Path(kagglehub.dataset_download('saurabhshahane/seoul-bike-trip-duration-prediction'))
print(f'Dataset cache: {dataset_dir}')

csv_matches = list(dataset_dir.glob('For_modeling.csv')) or list(dataset_dir.rglob('For_modeling.csv'))
if not csv_matches:
    print('A datasetben található CSV-k:')
    for c in sorted(dataset_dir.rglob('*.csv')):
        print(f'  - {c.relative_to(dataset_dir)} ({c.stat().st_size / 1e6:.1f} MB)')
    raise FileNotFoundError('For_modeling.csv nincs a Kaggle datasetben.')

csv_path = csv_matches[0]
print(f'CSV: {csv_path}  ({csv_path.stat().st_size / 1e6:.1f} MB)')

## 4. Verzió kiválasztása

Állítsd be a `VERSIONS` listát: melyik verziókat akarod most generálni. Az egészet újra is futtathatod — felülírja a meglévő fájlokat.

In [ ]:
# Itt kapcsolgasd: ['v0'], ['v0', 'vKNN'], vagy ['v0', 'vKNN', 'v1']
VERSIONS = ['v0']

for v in VERSIONS:
    assert v in SAMPLE_SIZES, f'ismeretlen verzió: {v}'
print(f'Generálandó verziók: {VERSIONS}')

## 5. Train/test parquet generálása (`prepare`)

CSV → `data/splits/train_{version}.parquet` + `test_{version}.parquet` (Drive-on `DATA_DIR/splits/`-be megy). v1 esetén ez ~10 perc lehet a 9,6M sor miatt.

In [ ]:
for v in VERSIONS:
    print(f'\n=== prepare({v!r}) ===')
    prepare(v, csv_path=csv_path)

## 6. Feature engineering + scaling (`build_features`)

Parquet → `data/X_{train,test}_{version}.npy` + `y_*.csv` + scaler joblib (`DATA_DIR`-be, azaz Drive-ra). **vKNN** esetén a KNN fit + R² kiértékelés is itt fut.

In [ ]:
results = {}
for v in VERSIONS:
    print(f'\n=== build_features({v!r}) ===')
    results[v] = build_features(v)

results

## 7. Ellenőrzés: betöltés a generált fájlokból

A `load_v1_data(version=...)` ugyanazt használja, amit a többi notebook. Itt csak shape-eket nézünk.

In [ ]:
from src.data_io import load_v1_data

for v in VERSIONS:
    X_tr, X_te, y_tr, y_te, names = load_v1_data(version=v)
    print(f'[{v}] X_train={X_tr.shape}  X_test={X_te.shape}  y_train={y_tr.shape}  features={len(names)}')